In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约3-5分钟）
!pip install -q numpy scipy matplotlib
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q openai-whisper
!pip install -q jieba
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 预下载Whisper模型
import whisper
print('正在下载Whisper small模型...')
model = whisper.load_model('small')
print('Whisper模型就绪！')


# 模块6 第2次课：ASR实战 + 端到端Pipeline整合本notebook包含：1. Whisper实战：安装、推理、API使用2. 实验：纯净语音 vs 带噪语音的识别率对比3. 实验：DeepFilterNet增强后语音的识别率4. GET声码器：从电极图还原语音5. 端到端Pipeline：带噪语音→增强→ACE→声码器→ASR6. 课程总结与展望---

In [ ]:
# 环境准备import numpy as npimport matplotlib.pyplot as pltimport os, sysprint("模块6 第2次课：ASR实战 + 端到端Pipeline整合")print()# 检查Whispertry:    import whisper    print("[OK] Whisper 可用")    WHISPER_AVAILABLE = Trueexcept ImportError:    print("[--] Whisper 不可用，请安装: pip install openai-whisper")    WHISPER_AVAILABLE = False# 检查ACE + GET声码器ACE_DIR = os.path.join('..', '..', 'module4-deepace', 'ACE')if os.path.exists(ACE_DIR):    sys.path.insert(0, ACE_DIR)    try:        from ace_strategy import ace_strategy        from get_voc import get_voc        print("[OK] ACE策略 + GET声码器 可用")        ACE_AVAILABLE = True    except ImportError as e:        print("[--] ACE/GET导入失败:", e)        ACE_AVAILABLE = Falseelse:    print("[--] ACE目录不存在:", ACE_DIR)    ACE_AVAILABLE = False# 检查DeepFilterNetDFN_DIR = os.path.join('..', '..', 'module5-deepfilternet', 'DeepFilterNet-main', 'DeepFilterNet')if os.path.exists(DFN_DIR):    sys.path.insert(0, DFN_DIR)    try:        from df.config import config        config.use_defaults()        print("[OK] DeepFilterNet 可用")        DF_AVAILABLE = True    except:        print("[--] DeepFilterNet 不可用")        DF_AVAILABLE = Falseelse:    DF_AVAILABLE = Falseprint()# 加载Whisper模型if WHISPER_AVAILABLE:    print("正在加载Whisper模型...")    try:        model = whisper.load_model("small")        print("[OK] Whisper small模型加载成功！")    except Exception as e:        print("[--] small模型加载失败:", e)        try:            model = whisper.load_model("tiny")            print("[OK] Whisper tiny模型加载成功")        except:            model = None            WHISPER_AVAILABLE = Falseelse:    model = None

## §1 Whisper实战### 1.1 基本使用Whisper的使用非常简单：1. 加载模型：`model = whisper.load_model("small")`2. 识别音频：`result = model.transcribe("audio.wav", language="zh")`3. 获取文本：`result["text"]`### 1.2 关键参数- `language`：指定语言（"zh"中文，"en"英文）- `task`："transcribe"（识别）或"translate"（翻译为英文）- `temperature`：解码温度，越高越随机- `beam_size`：Beam Search宽度

In [ ]:
# Whisper 识别演示if WHISPER_AVAILABLE and model is not None:    # 查找测试文件    test_dirs = [        os.path.join('test_audio'),        os.path.join('test_samples'),    ]    test_files = []    for td in test_dirs:        if os.path.exists(td):            for f in sorted(os.listdir(td)):                if f.endswith('.wav'):                    test_files.append(os.path.join(td, f))            break    if test_files:        # 优先选择clean文件        test_file = None        for f in test_files:            if 'clean' in os.path.basename(f).lower():                test_file = f                break        if test_file is None:            test_file = test_files[0]        print("识别文件:", os.path.basename(test_file))        print()        result = model.transcribe(test_file, language="zh", verbose=True)        print()        print("识别结果:", result["text"])    else:        print("未找到测试文件，跳过识别演示")else:    print("跳过识别演示（Whisper不可用）")

## §2 实验：噪声对ASR的影响### 假设噪声会降低ASR识别准确率。SNR越低，CER越高。### 实验设计对比不同SNR下的识别效果：- 干净语音 → ASR → CER（基线）- 0 dB SNR → ASR → CER- 5 dB SNR → ASR → CER- 10 dB SNR → ASR → CER

In [ ]:
# CER计算工具def levenshtein_distance(ref, hyp):    """计算编辑距离"""    n, m = len(ref), len(hyp)    dp = np.zeros((n + 1, m + 1), dtype=int)    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref[i-1] == hyp[j-1]:                dp[i][j] = dp[i-1][j-1]            else:                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])    return dp[n][m]def compute_cer(reference, hypothesis):    """计算字符错误率 (CER)"""    ref_chars = list(reference.replace(" ", ""))    hyp_chars = list(hypothesis.replace(" ", ""))    dist = levenshtein_distance(ref_chars, hyp_chars)    return dist / len(ref_chars) if len(ref_chars) > 0 else 0.0print("CER计算工具已就绪")

## §3 GET声码器：从电极图还原语音### 3.1 为什么需要声码器？ACE策略将语音转换为电极图（22个通道的刺激序列），但电极图不是音频信号，无法直接播放或送入ASR。**GET声码器**解决了这个问题：将电极图还原成可听的声学信号。```ACE电极图 (electrodogram)    |    |  GET声码器    |  - 每个电极 → 对应频率的正弦/噪声载波    |  - 刺激幅度 → 调制载波包络    |  - 高斯脉冲叠加    |  - 去加重滤波    |还原音频 (vocoded audio)```### 3.2 GET vs GEN| 参数 | GET (噪声带激励) | GEN (噪声激励) ||------|-----------------|---------------|| 载波类型 | 正弦波 | 带限噪声 || 音质 | 更清晰、更像语音 | 更粗糙 || 用途 | 研究中常用 | 替代方案 |### 3.3 声码器在Pipeline中的位置```带噪语音 → DeepFilterNet → 增强语音 → ACE → 电极图 → GET声码器 → 还原语音 → ASR                                                              ↑                                                        这一步是关键！```

In [ ]:
# GET声码器演示if ACE_AVAILABLE:    import numpy as np    from scipy.signal import resample_poly    from math import gcd    # 生成测试信号（模拟语音）    sr = 16000    duration = 2.0    t = np.linspace(0, duration, int(sr * duration), endpoint=False)    # 多谐波信号（模拟语音）    f0 = 150    clean = np.zeros_like(t)    for h in range(1, 8):        clean += (0.4 / h) * np.sin(2 * np.pi * f0 * h * t)    clean = clean / np.max(np.abs(clean)) * 0.36    # Step 1: ACE编码    print("Step 1: ACE编码...")    N_BAND = 22    N_MAXIMA = 8    q, p = ace_strategy(clean, sr, N_BAND, N_MAXIMA)    print("  电极数: %d" % N_BAND)    print("  每帧选通道: %d" % N_MAXIMA)    print("  脉冲总数: %d" % len(q['electrodes']))    print()    # Step 2: GET声码器还原    print("Step 2: GET声码器还原...")    GET_DURATIONS_FACTORS = (3 + (N_BAND - np.arange(1, N_BAND + 1))).astype(float)    GET_FS = 16000    vocoded, mod_bands = get_voc(        q, p,        vocoder_carrier=1,  # 1=GET(正弦), 2=GEN(噪声)        get_durations_factors=GET_DURATIONS_FACTORS,        conv_type=1,        carrier_freq_shift=0,        get_fs=GET_FS,    )    print("  还原音频长度: %.2f 秒" % (len(vocoded) / GET_FS))    print("  还原音频形状:", vocoded.shape)    print()    # 可视化对比    fig, axes = plt.subplots(3, 1, figsize=(12, 7))    # 原始信号    t_orig = np.arange(len(clean)) / sr    axes[0].plot(t_orig, clean)    axes[0].set_title("原始语音信号")    axes[0].set_xlabel("时间 (s)")    axes[0].set_ylabel("幅度")    # 电极图    ax_el = axes[1]    n_pulses = len(q['electrodes'])    pulse_times = np.arange(1, n_pulses + 1) * q['periods'] / 1e6    for idx in range(n_pulses):        el = int(q['electrodes'][idx])        if el == 0:            continue        ch = 23 - el        cl_norm = q['current_levels'][idx] / 255.0        ax_el.vlines(pulse_times[idx], ch, ch + cl_norm, colors='k', linewidth=0.5)    ax_el.set_title("ACE电极图")    ax_el.set_xlabel("时间 (s)")    ax_el.set_ylabel("电极")    # 还原信号    t_voc = np.arange(len(vocoded)) / GET_FS    axes[2].plot(t_voc, vocoded)    axes[2].set_title("GET声码器还原语音")    axes[2].set_xlabel("时间 (s)")    axes[2].set_ylabel("幅度")    plt.tight_layout()    plt.show()    print("GET声码器演示完成！")    print("还原音频虽然失真，但仍保留了语音的主要时频特征。")else:    print("跳过GET声码器演示（ACE模块不可用）")

## §4 端到端Pipeline整合### 4.1 完整Pipeline架构这是整个培训课程的最终目标——将所有模块串联：```┌──────────┐    ┌──────────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐│ 带噪语音  │ →  │ DeepFilterNet │ →  │   ACE    │ →  │ GET声码器 │ →  │  Whisper  │ → 文本│ (输入)    │    │  语音增强     │    │ CI编码   │    │ 语音还原  │    │  ASR识别  │└──────────┘    └──────────────┘    └──────────┘    └──────────┘    └──────────┘     ↓                  ↓                 ↓              ↓               ↓   原始音频          增强音频          电极图        还原音频         识别文本                  (模块5)          (模块4)       (模块4)          (模块6)```### 4.2 GET声码器的关键作用没有声码器，电极图只是数字序列，ASR无法处理。GET声码器将电极刺激模式还原为声学波形，使"增强→编码→识别"的评估闭环成为可能。### 4.3 评估对比| Pipeline配置 | 说明 | 预期效果 ||-------------|------|----------|| 带噪→ASR | 无处理基线 | CER最高 || 增强→ASR | 只做语音增强 | CER降低 || 增强→ACE→声码器→ASR | 完整CI pipeline | CER中等（声码器损失信息）|| 干净→ASR | 上界参考 | CER最低 || 干净→ACE→声码器→ASR | 纯净语音经CI处理 | CER中等（ACE+声码器损失）|

In [ ]:
# 端到端Pipeline完整实现def full_pipeline(audio, sr=16000, use_enhancement=False, use_ace_vocoder=False):    """    完整CI语音处理Pipeline。    参数:        audio: 输入音频 (numpy array)        sr: 采样率        use_enhancement: 是否使用DeepFilterNet增强        use_ace_vocoder: 是否经过ACE+GET声码器    返回:        processed_audio: 处理后的音频        stages: 经过的处理阶段列表    """    stages = []    processed = audio.copy()    # ===== Step 1: 语音增强（可选）=====    if use_enhancement:        stages.append("DeepFilterNet增强")        # 实际代码：        # from df.enhance import init_df, enhance        # model_df, df_state, _, _ = init_df()        # enhanced = enhance(model_df, df_state, torch.tensor(processed))        # processed = enhanced.numpy()    # ===== Step 2: ACE编码 + GET声码器还原（可选）=====    if use_ace_vocoder:        stages.append("ACE编码")        stages.append("GET声码器还原")        if ACE_AVAILABLE:            # 重采样到16kHz            if sr != 16000:                from scipy.signal import resample_poly                from math import gcd                g = gcd(16000, sr)                processed = resample_poly(processed, 16000 // g, sr // g)                sr = 16000            # 归一化            processed = processed / (np.max(np.abs(processed)) + 1e-8) * 0.36            # ACE编码            q, p = ace_strategy(processed, 16000, 22, 8)            # GET声码器还原            GET_DUR = (3 + (22 - np.arange(1, 23))).astype(float)            processed, _ = get_voc(q, p, 1, GET_DUR, 1, 0, 16000)            sr = 16000    return processed, sr, stages# ===== 运行对比实验 =====print("=== Pipeline对比实验（模拟结果）===")print()configs = [    ("带噪→ASR (无处理)",                    False, False, 0.45),    ("增强→ASR",                             True,  False, 0.15),    ("增强→ACE→GET声码器→ASR",               True,  True,  0.35),    ("干净→ASR (上界)",                      False, False, 0.05),    ("干净→ACE→GET声码器→ASR (CI上界)",      False, True,  0.30),]print("%-35s %-10s %s" % ("配置", "CER", "效果"))print("-" * 70)for name, enh, ace, cer in configs:    bar = "█" * int(cer * 30) + "░" * (30 - int(cer * 30))    print("  %-33s %.2f  %s" % (name, cer, bar))print()print("注意：以上CER数值为示意值，实际结果取决于音频内容。")print("关键观察：ACE+声码器会损失信息（CER从0.15→0.35），但增强仍有帮助。")

## §5 GET声码器深入分析### 5.1 信息损失来源从原始语音到声码器还原，信息在每个阶段都有损失：| 处理阶段 | 信息损失 | 原因 ||---------|---------|------|| ACE编码 | 频率分辨率 | 22通道 vs 原始频谱 || ACE编码 | 通道选择 | n-of-m策略丢弃部分通道 || GET声码器 | 相位信息 | 正弦载波无原始相位 || GET声码器 | 频率精度 | 通道中心频率的近似 |### 5.2 为什么声码器还原的语音CER更高？1. **频率分辨率降低**：22通道无法还原原始频谱的精细结构2. **通道选择丢弃信息**：Nmaxima=8意味着每帧只保留8/22的通道3. **包络近似**：高斯脉冲叠加是对实际刺激的简化模拟4. **无原始相位**：声码器使用正弦载波，丢失了原始语音的相位信息### 5.3 声码器还原音频的听觉特征- 听起来像"机器人"语音- 音调可辨但音色不同- 噪声成分增加- 对于CI用户研究来说，这模拟了"CI用户通过声码器听到的声音"（正常听力者模拟CI体验）> **研究意义**：GET声码器也常用于"声码器模拟"实验——让正常听力者体验类似CI的听觉感受，评估CI处理策略的效果。

In [ ]:
# 对比不同阶段音频的频谱（需要ACE可用）if ACE_AVAILABLE:    import numpy as np    import matplotlib.pyplot as plt    sr = 16000    duration = 2.0    t = np.linspace(0, duration, int(sr * duration), endpoint=False)    # 生成信号    f0 = 150    clean = np.zeros_like(t)    for h in range(1, 8):        clean += (0.4 / h) * np.sin(2 * np.pi * f0 * h * t)    clean = clean / np.max(np.abs(clean)) * 0.36    # ACE + GET    q, p = ace_strategy(clean, sr, 22, 8)    GET_DUR = (3 + (22 - np.arange(1, 23))).astype(float)    vocoded, _ = get_voc(q, p, 1, GET_DUR, 1, 0, 16000)    # 频谱对比    fig, axes = plt.subplots(2, 1, figsize=(12, 6))    # 短时傅里叶变换    from scipy.signal import stft    f1, t1, Z1 = stft(clean, fs=sr, nperseg=256, noverlap=192)    f2, t2, Z2 = stft(vocoded[:len(clean)], fs=sr, nperseg=256, noverlap=192)    axes[0].pcolormesh(t1, f1, 20*np.log10(np.abs(Z1)+1e-8),                       shading='auto', cmap='magma', vmin=-60, vmax=0)    axes[0].set_title("原始信号频谱")    axes[0].set_ylabel("频率 (Hz)")    axes[1].pcolormesh(t2, f2, 20*np.log10(np.abs(Z2)+1e-8),                       shading='auto', cmap='magma', vmin=-60, vmax=0)    axes[1].set_title("GET声码器还原频谱 (ACE 22通道, Nmaxima=8)")    axes[1].set_ylabel("频率 (Hz)")    axes[1].set_xlabel("时间 (s)")    plt.tight_layout()    plt.show()    print("观察：GET声码器还原的频谱中，高频细节丢失，频率分辨率降低。")else:    print("跳过频谱对比（ACE模块不可用）")

## §6 各模块知识回顾### 培训课程知识图谱```模块0: Python编程基础  ↓ (编程能力)模块1: Linux + 深度学习环境  ↓ (环境就绪)模块2: 深度学习入门  ↓ (神经网络、CNN、训练技巧)模块3: 面向声音的分类模型  ↓ (音频特征、CRNN、CI分类任务)模块4: DeepACE模型解析  ↓ (论文精读、ACE编码、GET声码器)模块5: DeepFilterNet模型解析  ↓ (语音增强、ERB域、评估指标)模块6: ASR + Pipeline整合 ← 你在这里```### 完整Pipeline中的模块对应| Pipeline阶段 | 对应模块 | 核心知识 ||-------------|---------|----------|| 语音增强 | 模块5 DeepFilterNet | 两阶段增强、ERB域、SI-SDR || CI编码 | 模块4 ACE策略 | 通道选择、电极映射 || 语音还原 | 模块4 GET声码器 | 正弦载波、高斯包络 || 语音识别 | 模块6 Whisper | CTC/Attention、WER/CER |

---## 小结与课程回顾本节课我们：1. 实战使用了Whisper进行中文语音识别2. 分析了噪声对ASR识别率的影响3. 学习了GET声码器将电极图还原为语音的方法4. 整合了完整的端到端Pipeline（含声码器步骤）5. 分析了声码器还原中的信息损失6. 回顾了整个培训课程的知识体系**培训课程总体回顾**：从Python编程基础开始，经过深度学习入门、音频分类、CI编码策略（含GET声码器）、语音增强，最终整合为完整的CI语音处理Pipeline。**继续深入的方向**：- 深入研究DeepACE/DeepFilterNet的修改实验- 学习模型训练技术（微调、迁移学习）- 探索CI专用语音增强方法- 改进GET声码器的音质（更多通道、更好的载波）- 开展CI用户主观评估实验- 阅读更多相关论文